In [ ]:
from google.colab import drive
import os

# 1. Driveをマウント
drive.mount('/content/drive')

# 2. ログ保存用のフォルダを作成
LOG_DIR = "/content/drive/MyDrive/Research_Logs"
os.makedirs(LOG_DIR, exist_ok=True)

print(f"✅ ログ保存先: {LOG_DIR}")

In [ ]:
import json
import pandas as pd
from pathlib import Path

def analyze_kv_results(file_path: str):
    data = []

    # 1. ファイルの読み込み
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))

    # 2. DataFrameに変換して分析しやすくする
    df = pd.DataFrame(data)

    # 3. 正解判定 (単純一致チェック)
    df["is_correct"] = df.apply(
        lambda r: str(r["value"]).strip().lower() in str(r["output"]).strip().lower(),
        axis=1
    )

    # 4. 統計情報の表示
    total = len(df)
    accuracy = df["is_correct"].mean() * 100

    print("--- Experiment Summary ---")
    print(f"Total Samples: {total}")
    print(f"Overall Accuracy: {accuracy:.2f}%")
    print(f"Avg Num KV: {df['num_kv'].mean():.1f}")

    # 5. 位置別の正解率 (Lost-in-the-Middle の確認に重要)
    print("\n--- Accuracy by Target Index ---")
    # target_index を 5 等分などのビンに分けると傾向が見やすいですが、まずはそのまま表示
    pos_acc = df.groupby("target_index")["is_correct"].mean() * 100
    print(pos_acc.to_string(float_format="{:.1f}%".format))

    # 6. 失敗例のサンプリング (デバッグ用)
    errors = df[df["is_correct"] == False]
    if not errors.empty:
        print("\n--- Error Samples (First 3) ---")
        for i, row in errors.head(3).iterrows():
            print(f"Key: {row['key']}")
            print(f"Expected: {row['value']}")
            print(f"Actual Output: {row['output']}")
            print(f"Index: {row['target_index']} / Total: {row['num_kv']}")
            print("-" * 20)

    return df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_results(df):
    """
    分析結果をグラフ化する関数
    """
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(12, 5))

    # --- 1. 位置別の正解率 (Line Plot) ---
    plt.subplot(1, 2, 1)
    # target_indexごとの平均正解率を算出
    pos_acc = df.groupby("target_index")["is_correct"].mean() * 100
    sns.lineplot(x=pos_acc.index, y=pos_acc.values, marker='o')
    plt.title("Accuracy by Target Index\n(Lost-in-the-Middle Check)")
    plt.xlabel("Target Index (Position)")
    plt.ylabel("Accuracy (%)")
    plt.ylim(-5, 105)

    # --- 2. KV数（文脈長）別の正解率 (Bar Plot) ---
    plt.subplot(1, 2, 2)
    if df['num_kv'].nunique() > 1:
        # 複数の文脈長（num_kv）がある場合
        len_acc = df.groupby("num_kv")["is_correct"].mean() * 100
        sns.barplot(x=len_acc.index, y=len_acc.values, palette="viridis")
        plt.title("Accuracy by Context Length")
        plt.xlabel("Number of KV Pairs")
        plt.ylabel("Accuracy (%)")
        plt.ylim(-5, 105)
    else:
        # 文脈長が一定の場合は、正解・不正解のカウントを表示
        sns.countplot(x="is_correct", data=df, palette="Set2")
        plt.title("Correct vs Incorrect Counts")
        plt.xlabel("Is Correct")
        plt.ylabel("Count")

    plt.tight_layout()
    plt.show()

    # --- 3. (オプション) ヒートマップ ---
    # もし複数の num_kv と target_index が混在している場合、
    # どこで間違えやすいかを可視化できます
    if df['num_kv'].nunique() > 1:
        plt.figure(figsize=(10, 6))
        pivot_df = df.pivot_table(index="num_kv", columns="target_index", values="is_correct", aggfunc="mean")
        sns.heatmap(pivot_df, annot=True, fmt=".2f", cmap="RdYlGn", cbar_kws={'label': 'Accuracy'})
        plt.title("Accuracy Heatmap (Context Length vs Position)")
        plt.show()

# --- 使用例 ---
# 既存の analyze_kv_results の最後で呼び出すか、以下のように実行します
if __name__ == "__main__":
    PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/kv_retrieval/output_kv_retrieval_scale.jsonl"
    if Path(PATH).exists():
        df_results = analyze_kv_results(PATH)
        # グラフ表示を実行
        visualize_results(df_results)
    PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/kv_retrieval/output_kv_retrieval_strong_scale.jsonl"
    if Path(PATH).exists():
        df_results = analyze_kv_results(PATH)
        # グラフ表示を実行
        visualize_results(df_results)

    else:
        print(f"File not found: {PATH}")

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def load_results(file_path):
    """JSONLファイルを読み込み、NLLからPPLを復元してDataFrameを作成"""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                # PPLをavg_nllから復元 (exp(nll))
                record['ppl'] = np.exp(record['avg_nll'])
                data.append(record)
            except Exception as e:
                continue
    return pd.DataFrame(data)

def analyze_kv_results(df, label="Experiment"):
    print(f"\n{'='*20} Analysis: {label} {'='*20}")

    # 1. 全体統計（加重平均の計算）
    # 全体のPPLは、各サンプルのPPLの平均ではなく、
    # 全トークンのNLL合計を総トークン数で割ったものの exp をとる
    df['total_nll'] = df['avg_nll'] * df['target_len']
    total_tokens = df['target_len'].sum()

    corpus_avg_nll = df['total_nll'].sum() / total_tokens
    corpus_ppl = np.exp(corpus_avg_nll)

    print(f"Total Samples: {len(df)}")
    print(f"Weighted Avg NLL: {corpus_avg_nll:.4f}")
    print(f"Corpus PPL:       {corpus_ppl:.4f}")

    # 2. 位置別の分析
    # 正解の位置を 0(最初) ~ 1(最後) に正規化
    df['relative_position'] = df['target_index'] / (df['num_kv'] - 1)

    # 5つの区間に分割
    df['pos_bin'] = pd.cut(df['relative_position'], bins=5,
                           labels=['0-20%', '20-40%', '40-60%', '60-80%', '80-100%'])

    # 各区間での平均PPLを算出
    bin_stats = df.groupby('pos_bin', observed=True)['ppl'].mean()

    print("\n[PPL by Relative Position]")
    print(bin_stats)

    return corpus_ppl, bin_stats

def plot_kv_comparison(results_dict):
    """複数実験の Lost-in-the-Middle グラフを描画"""
    plt.figure(figsize=(10, 6))

    for label, bin_stats in results_dict.items():
        plt.plot(bin_stats.index, bin_stats.values, marker='o', label=label)

    plt.title("KV Retrieval: Impact of Information Position on PPL")
    plt.xlabel("Relative Position in Context")
    plt.ylabel("Average Perplexity (PPL)")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

# --- 使用例 (実験終了後に実行) ---
path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/kv_retrieval/nll_kv_retrieval_no_scale.jsonl"
df_no_scale = load_results(path)
path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/kv_retrieval/nll_kv_retrieval_scale.jsonl"
df_scale = load_results(path)

ppl_base, bins_base = analyze_kv_results(df_no_scale, "Baseline")
ppl_scaled, bins_scaled = analyze_kv_results(df_scale, "Scaling (-10)")

plot_kv_comparison({"Baseline": bins_base, "Scaling (-10)": bins_scaled})